# 7. Text preprocessing for Persian NLP

## Small idea: preserve evidence before normalizing it

Text cleaning should not erase the phenomena a study may later need. Keep at least a raw
view and a reproducible normalized view. An optional analysis view can segment URLs or
mentions while retaining emojis, hashtags, punctuation, code-mixing, elongation, and
other potentially meaningful signals.

**Learning goals**

- preserve aligned raw, normalized, and analysis-oriented text views;
- harmonize common Persian/Arabic character variants and spacing;
- detect exact and near duplicates before splitting;
- fit word and character TF–IDF vectorizers on training text only;
- avoid destructive “cleaning” defaults.

In [ ]:
import hashlib
import re
import unicodedata
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

## 1. Build aligned text views

In [ ]:
ARABIC_TO_PERSIAN = str.maketrans({
    "ي": "ی",
    "ى": "ی",
    "ك": "ک",
})
INVISIBLE_PATTERN = re.compile(r"[\u200e\u200f\u202a-\u202e\u2066-\u2069]")
SPACE_PATTERN = re.compile(r"[ \t\r\f\v]+")
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
MENTION_PATTERN = re.compile(r"(?<!\w)@[\w_]+", flags=re.UNICODE)

def normalize_persian(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text))
    text = text.translate(ARABIC_TO_PERSIAN)
    text = INVISIBLE_PATTERN.sub("", text)
    text = re.sub(r"\s*\u200c\s*", "\u200c", text)
    text = SPACE_PATTERN.sub(" ", text)
    return text.strip()

def analysis_view(text: str) -> str:
    text = normalize_persian(text)
    text = URL_PATTERN.sub(" <URL> ", text)
    text = MENTION_PATTERN.sub(" <USER> ", text)
    return SPACE_PATTERN.sub(" ", text).strip()

In [ ]:
texts = pd.DataFrame({
    "document_id": ["D1", "D2", "D3", "D4", "D5", "D6"],
    "text_raw": [
        "من كتاب فارسي را دوست دارم",
        "من کتاب فارسی را دوست دارم",
        "این تمرین خیلییی سخت بود 😅!!!",
        "نظر من در https://example.org هست",
        "@user من موافقم #فارسی",
        "نمی ‌ دانم چرا...",
    ],
})

texts["text_normalized"] = texts["text_raw"].map(normalize_persian)
texts["text_analysis"] = texts["text_raw"].map(analysis_view)
texts

The raw column remains unchanged. The normalized view harmonizes Unicode and spacing. The
analysis view segments URLs and user mentions but deliberately retains emoji, hashtags,
elongation, punctuation, and code-mixing.

## 2. Exact duplicates after normalization

In [ ]:
texts["normalized_hash"] = texts["text_normalized"].map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]
)

exact_duplicate_mask = texts.duplicated("normalized_hash", keep=False)
texts.loc[exact_duplicate_mask, ["document_id", "text_raw", "text_normalized", "normalized_hash"]]

## 3. A small near-duplicate inspection

In [ ]:
pairs = []
for left in range(len(texts)):
    for right in range(left + 1, len(texts)):
        similarity = SequenceMatcher(
            None,
            texts.loc[left, "text_normalized"],
            texts.loc[right, "text_normalized"],
        ).ratio()
        if similarity >= 0.80:
            pairs.append({
                "left": texts.loc[left, "document_id"],
                "right": texts.loc[right, "document_id"],
                "similarity": round(similarity, 3),
            })

pd.DataFrame(pairs)

Pairwise comparison is suitable only for tiny teaching data. Large corpora need blocking,
MinHash, locality-sensitive hashing, or another scalable candidate-generation method.
Confirm clusters manually when the distinction between duplicate and legitimate formulaic
language matters. Keep each duplicate cluster in one data partition.

## 4. Fit word TF–IDF on training text only

In [ ]:
train_texts = texts.loc[:3, "text_normalized"]
future_texts = texts.loc[4:, "text_normalized"]

word_vectorizer = TfidfVectorizer(
    preprocessor=normalize_persian,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)

X_train_word = word_vectorizer.fit_transform(train_texts)
X_future_word = word_vectorizer.transform(future_texts)

print("Word vocabulary size:", len(word_vectorizer.vocabulary_))
print("Shapes:", X_train_word.shape, X_future_word.shape)
print("First features:", word_vectorizer.get_feature_names_out()[:12].tolist())

The vocabulary and inverse-document-frequency weights come only from training text. Do not
fit a vectorizer on the full corpus before splitting.

## 5. Character n-grams as an orthographic baseline

In [ ]:
character_vectorizer = TfidfVectorizer(
    preprocessor=normalize_persian,
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    sublinear_tf=True,
)

X_train_char = character_vectorizer.fit_transform(train_texts)
X_future_char = character_vectorizer.transform(future_texts)

print("Character vocabulary size:", len(character_vectorizer.vocabulary_))
print("Shapes:", X_train_char.shape, X_future_char.shape)

Character n-grams can capture affixes, spacing variation, and spelling patterns without a
full tokenizer. They can also memorize names, templates, or source artifacts, so duplicate
control and error analysis remain essential.

## What not to remove automatically

- negation and function words;
- نیم‌فاصله distinctions;
- emojis, hashtags, and punctuation;
- repetitions such as `خیلییی`;
- code-mixed words;
- diacritics when they are relevant to the research question.

Each removal should follow the task, corpus, and evaluation plan—not a generic cleaning list.

## Tiny checkpoint

Add two texts that differ only in Arabic/Persian `ی` or `ک`, then confirm that their
normalized hashes match. Next compare word and character feature counts.

## References

- scikit-learn text feature extraction: https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
- `TfidfVectorizer`: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- Unicode normalization: https://docs.python.org/3/library/unicodedata.html